# Claude Groups 1–6 analysis validation

## tl;dr

The analysis is directionally strong but needs revision before it becomes a decision record. The
G3/G5 worker confound and the target-KL failure are real. Material corrections are: use paired
scenario inference instead of a universal ±4.2/8-collision rule; align rollout KPIs to
`rollout_policy_update`, not the same-number post-update checkpoint; change target-KL 0.02 early
stops from 10/20 to 11/20; and describe G5 clip 0.20 as non-monotonic.


## Context & Methods

This notebook is a diagnostic companion. Its source calculation is
`validate_claude_analysis.py`, which reads the recorded `run_config.json`, `metrics.jsonl`,
`episodes.jsonl`, actor checkpoints, and Austin600 `results_multi.json` files.

### Key Assumptions

- Austin600 is a fixed paired scenario panel. Exact paired p-values are descriptive and unadjusted
  for checkpoint selection; they do not establish cross-seed or out-of-panel generalization.
- `ego_collision=true` with `opp_collision=false` is classified as ego/wall-like, matching the
  simulator collision flags; it is not a full geometric root-cause label.
- The worker-count association is verified, while the internal source of process-topology
  sensitivity remains unresolved.


In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path(globals().get("_REPO_ROOT", Path.cwd()))
if not (ROOT / "analysis_results" / "ppo_groups_1_6_validation").exists():
    raise FileNotFoundError("Run from the End2Race repository root")
OUT = ROOT / "analysis_results" / "ppo_groups_1_6_validation"

summary = json.loads((OUT / "validation_summary.json").read_text())
claims = pd.read_csv(OUT / "claim_review.csv")
target_steps = pd.read_csv(OUT / "target_kl_steps.csv")
target_eval = pd.read_csv(OUT / "target_kl_eval.csv")
paired = pd.read_csv(OUT / "paired_scenario_tests.csv")
actor_diff = pd.read_csv(OUT / "old_vs_long_actor_diff.csv")

print(summary["overall_assessment"])
print(f"Claims reviewed: {len(claims)}; high-severity issues: {(claims.severity == 'high').sum()}")


needs revision before being used as a decision record; directionally strong but contains two high-severity methodology errors and several causal overclaims
Claims reviewed: 10; high-severity issues: 2


## Data

### 1. Claim-level audit

The table separates numerical discrepancies from causal or methodological overreach.


In [2]:
print(claims[["claim", "assessment", "severity", "required_revision"]].to_string(index=False))


                                                                              claim                 assessment severity                                                                                                                               required_revision
                                       G1 privilege_gru is the strongest critic arm                  supported     none               Remove the causal phrase that independent_gru gives harmful advantage directions; the eval path does not identify that mechanism.
            Batch 12800 is best and larger batches also reduce optimizer-step count      supported with nuance      low                        Call it the effect of changing batch under fixed epochs; it does not isolate batch aggregation from optimization budget.
                     G3 old clip runs are confounded against the 12-worker baseline                  supported     none                                                    Keep the planner shallow-copy explana

## Results

### 2. G3 versus G5 diverges before PPO actor updates

The old G3 and G5 long clip 0.20 configs differ in `env_workers` and total horizon. Total horizon
does not alter the constant LR/clip schedules before U20; the first warm-up rollout already differs,
so the observed model divergence cannot be caused by clip or later checkpoints.


In [3]:
g3 = summary["g3_vs_g5"]
print("Config differences:", json.dumps(g3["config_differences"], indent=2))
print("Old warm-up:", g3["old_warmup"])
print("Long warm-up:", g3["long_warmup"])
print("First structural difference:", g3["first_structural_difference"])
print("\nActor checkpoint differences:")
print(actor_diff.to_string(index=False))


Config differences: {
  "env_workers": {
    "old_g3": 8,
    "g5_long": 12
  },
  "num_updates": {
    "old_g3": 20,
    "g5_long": 30
  },
  "output_dir": {
    "old_g3": "post-trained/ppo_privilege_gru_0721_clip020",
    "g5_long": "post-trained/ppo_privilege_gru_0722_long_clip020"
  }
}
Old warm-up: {'epochs': 19, 'best_epoch': 16, 'best_validation_loss': 0.1696825691263307, 'rollout1_episode_count': 152}
Long warm-up: {'epochs': 5, 'best_epoch': 2, 'best_validation_loss': 0.29815395020017543, 'rollout1_episode_count': 153}
First structural difference: {'row': 3, 'old_scenario_id': 'collision-sp003-ego0073-raceline2-i15-v080', 'long_scenario_id': 'collision-sp003-ego0073-raceline2-i15-v080', 'old_steps': 623, 'long_steps': 628, 'old_outcome': 'ego_collision', 'long_outcome': 'ego_collision'}

Actor checkpoint differences:
 update  keys_match  tensor_equal  max_abs_parameter_difference
      1        True         False                      0.000571
      5        True         False 

### 3. Target-KL changes the optimization path, not just a scalar metric

The gate is evaluated before the current minibatch update. A large KL at the gate was produced by
earlier completed minibatches; stopping cannot undo those steps. Completed steps therefore vary
sharply by update while the critic still receives its five epochs.


In [4]:
step_summary = target_steps.groupby("label").agg(
    early_stops=("early_stop", "sum"),
    completed_steps=("steps_completed", "sum"),
    planned_steps=("steps_planned", "sum"),
    max_trigger_kl=("trigger_kl", "max"),
)
print(step_summary.to_string())
print("\nTarget-KL 0.04 update path:")
print(target_steps[target_steps.label == "target-KL 0.04"][["update", "steps_completed", "early_stop", "trigger_kl"]].to_string(index=False))
print("\nTarget-KL 0.04 evaluation path:")
print(target_eval.round(4).to_string(index=False))


                early_stops  completed_steps  planned_steps  max_trigger_kl
label                                                                      
target-KL 0.02           11              206            320        2.426827
target-KL 0.04           12              203            320        1.050665

Target-KL 0.04 update path:
 update  steps_completed  early_stop  trigger_kl
      1                1        True    0.077282
      2                1        True    0.076644
      3               16       False         NaN
      4               14        True    0.072763
      5               16       False         NaN
      6               16       False         NaN
      7               16       False         NaN
      8                2        True    0.619845
      9               16       False         NaN
     10               16       False         NaN
     11                4        True    0.505776
     12                9        True    0.744343
     13                3      

### 4. Paired scenario evidence replaces a universal count threshold

The relevant observation is each scenario's before/after outcome. The p-values below are exact
two-sided McNemar/binomial tests on discordant pairs and are unadjusted for selecting among several
checkpoints.


In [5]:
print(paired.round(4).to_string(index=False))


                       comparison  before_collisions  after_collisions  shared  resolved  created  net_change  paired_exact_p_unadjusted
                   BC -> base U20                 22                14      12        10        2          -8                     0.0386
       BC -> old G3 clip 0.20 U20                 22                14       4        18       10          -8                     0.1849
      BC -> G5 long clip 0.20 U30                 22                11       5        17        6         -11                     0.0347
base U20 -> G5 long clip 0.20 U30                 14                11       5         9        6          -3                     0.6072
         BC -> target-KL 0.04 U20                 22                33       8        14       25          11                     0.1081


## Takeaways

1. Keep the direction of the six-group conclusion, but revise the statistical and temporal-alignment claims.
2. Treat `env_workers` as an experiment parameter until a process-isolation test proves invariance.
3. Use Group 5—not legacy Group 3—as the authoritative clip 0.15 versus 0.20 comparison.
4. Disable target-KL in the selected recipe (`None`), but retain the optional implementation and telemetry.
5. Describe mechanism claims such as harmful advantages, conservative reward gaming, planner shallow-copy
   contamination, and near-impossible scenarios as hypotheses until directly tested.
